# Lab 2: Building RNN-Based Models

**Course:** Large Language Models — From Theory to Production  
**Part 2:** From Recurrent Networks to Transformers  
**Author:** Bassem Ben Hamed — ENETCOM, University of Sfax  

---

## Objectives

In this lab, we will put into practice the theoretical concepts from Section 2.1 by:

1. **Implementing a text classification model with LSTM** — Sentiment analysis on IMDB reviews
2. **Building a Seq2Seq model for a simple translation task** — English-to-French number translation
3. **Visualizing hidden states and gradient flow** — Understanding what RNNs learn internally
4. **Understanding the limitations of RNNs on long sequences** — Empirical evidence of vanishing gradients

**Tools:** PyTorch, Hugging Face Datasets, Matplotlib, NumPy

**Prerequisites:** Basic Python, familiarity with PyTorch tensors and `nn.Module`

## 0. Environment Setup

We start by installing the required libraries and checking GPU availability. If you are running this on **Google Colab**, make sure to enable a GPU runtime via `Runtime > Change runtime type > T4 GPU`.

In [ ]:
# Install dependencies (uncomment if needed, e.g. on Colab)
# !pip install torch torchtext datasets matplotlib numpy tqdm -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter
from tqdm.auto import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

# Part 1: Text Classification with LSTM

## 1.1 Theoretical Recap

### Vanilla RNN

A vanilla RNN computes a hidden state $h_t$ at each time step by combining the previous hidden state $h_{t-1}$ with the current input $x_t$:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)$$

**Problem:** During backpropagation through time (BPTT), the gradient of the loss with respect to early hidden states involves repeated multiplication by $W_{hh}$. If its eigenvalues are < 1, gradients **vanish**; if > 1, they **explode**. This makes it very hard for vanilla RNNs to learn **long-range dependencies**.

### LSTM (Long Short-Term Memory)

LSTM solves the vanishing gradient problem by introducing a **cell state** $C_t$ (a highway for information) and three **gates** that regulate information flow:

| Gate | Formula | Role |
|------|---------|------|
| **Forget** | $f_t = \sigma(W_f \cdot [h_{t-1}, x_t])$ | What to **discard** from cell state |
| **Input** | $i_t = \sigma(W_i \cdot [h_{t-1}, x_t])$ | What **new info** to store |
| **Output** | $o_t = \sigma(W_o \cdot [h_{t-1}, x_t])$ | What to **output** from cell state |

**Candidate cell state** (new information to potentially add):

$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t])$$

**Cell state update** (the key equation — additive, not multiplicative!):

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Hidden state output:**

$$h_t = o_t \odot \tanh(C_t)$$

The **additive** nature of the cell state update ($C_t = f_t \odot C_{t-1} + ...$) is what prevents the vanishing gradient — gradients can flow through the cell state unchanged when $f_t \approx 1$.

### Our Task: Sentiment Classification

We will build an LSTM that reads an IMDB movie review token by token, and uses the **final hidden state** $h_T$ to predict the sentiment (positive or negative). The architecture is:

```
Input tokens → Embedding → LSTM → Final hidden state → Linear → Sigmoid → P(positive)
```

## 1.2 Load and Explore the IMDB Dataset

We use the **IMDB dataset** from Hugging Face — 50,000 movie reviews (25k train, 25k test), each labeled as positive (1) or negative (0).

In [ ]:
from datasets import load_dataset

dataset = load_dataset('imdb')
print(f"Train: {len(dataset['train'])} examples")
print(f"Test:  {len(dataset['test'])} examples")
print(f"\nExample review (first 300 chars):")
print(f"Text:  {dataset['train'][0]['text'][:300]}...")
print(f"Label: {dataset['train'][0]['label']} ({'positive' if dataset['train'][0]['label'] == 1 else 'negative'})")

## 1.3 Tokenization and Vocabulary Building

Before feeding text to our LSTM, we need to:
1. **Tokenize** each review into words (simple whitespace + punctuation split)
2. **Build a vocabulary** mapping each word to an integer index
3. **Numericalize** each review into a sequence of indices

We limit the vocabulary to the top 25,000 most frequent words. Rare words are replaced by a special `<unk>` token. We also add a `<pad>` token for padding sequences to equal length in a batch.

In [ ]:
# --- Simple tokenizer ---
def tokenize(text):
    """Lowercase and split on non-alphanumeric characters."""
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

# --- Build vocabulary from training data ---
MAX_VOCAB = 25_000

counter = Counter()
for example in tqdm(dataset['train'], desc='Building vocab'):
    counter.update(tokenize(example['text']))

# Special tokens
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
PAD_IDX = 0
UNK_IDX = 1

# Build word-to-index mapping
vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for word, _ in counter.most_common(MAX_VOCAB - 2):  # -2 for special tokens
    vocab[word] = len(vocab)

# Reverse mapping for later inspection
idx_to_word = {idx: word for word, idx in vocab.items()}

print(f"Vocabulary size: {len(vocab):,}")
print(f"Top 10 words: {list(vocab.keys())[2:12]}")
print(f"Coverage: {sum(counter[w] for w in vocab) / sum(counter.values()):.1%} of tokens")

In [ ]:
# --- Numericalize: convert text to index sequences ---
def numericalize(text, max_len=256):
    """Tokenize and convert to indices, truncating to max_len."""
    tokens = tokenize(text)
    indices = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    return indices

# Test
sample = "This movie was absolutely fantastic!"
indices = numericalize(sample)
print(f"Text:    {sample}")
print(f"Tokens:  {tokenize(sample)}")
print(f"Indices: {indices}")
print(f"Decoded: {[idx_to_word[i] for i in indices]}")

## 1.4 PyTorch Dataset and DataLoader

We wrap our data in a PyTorch `Dataset` and use a custom `collate_fn` that:
- Pads all sequences in a batch to the same length (the longest in the batch)
- Returns the actual lengths so we can use **packed sequences** later (which tell the LSTM to ignore padding tokens)

Using packed sequences is important because without them, the LSTM would process `<pad>` tokens and pollute the hidden state.

In [ ]:
MAX_LEN = 256  # Truncate reviews longer than this

class IMDBDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = []
        for example in tqdm(hf_dataset, desc='Numericalizing', leave=False):
            indices = numericalize(example['text'], max_len=MAX_LEN)
            if len(indices) > 0:
                self.data.append((torch.tensor(indices, dtype=torch.long),
                                  example['label']))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    """Pad sequences and return lengths for packing."""
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts])
    # Pad to the longest sequence in this batch
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(labels, dtype=torch.float)
    return texts_padded, labels, lengths


# Build datasets
train_dataset = IMDBDataset(dataset['train'])
test_dataset  = IMDBDataset(dataset['test'])

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn)

# Inspect a batch
batch_texts, batch_labels, batch_lengths = next(iter(train_loader))
print(f"Batch text shape:   {batch_texts.shape}  (batch_size x max_seq_len)")
print(f"Batch labels shape: {batch_labels.shape}")
print(f"Sequence lengths:   min={batch_lengths.min()}, max={batch_lengths.max()}, mean={batch_lengths.float().mean():.0f}")

## 1.5 LSTM Classifier Model

Our model architecture:

| Layer | Description | Output Shape |
|-------|-------------|--------------|
| `Embedding` | Learns a dense vector for each word | `(batch, seq_len, emb_dim)` |
| `LSTM` | Processes the sequence, returns all hidden states | `(batch, seq_len, hidden_dim * num_directions)` |
| `Dropout` | Regularization on the final hidden state | `(batch, hidden_dim * num_directions)` |
| `Linear` | Maps hidden state to 1 logit (positive/negative) | `(batch, 1)` |

**Key design choices:**
- **Bidirectional LSTM:** Reads the sequence both forward and backward, giving the model context from both directions. The final representation is the **concatenation** of the last forward hidden state and the first backward hidden state: $h = [\overrightarrow{h_T} ; \overleftarrow{h_1}]$.
- **Packed sequences:** We use `pack_padded_sequence` to avoid processing padding tokens.
- **Pre-trained embeddings are optional** (we train from scratch here for simplicity).

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim,
                 n_layers, bidirectional, dropout, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )

        num_directions = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * num_directions, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, lengths):
        # text: (batch, seq_len)
        embedded = self.dropout(self.embedding(text))  # (batch, seq_len, emb_dim)

        # Pack to ignore padding
        packed = pack_padded_sequence(embedded, lengths.cpu(),
                                     batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed)
        # hidden: (n_layers * n_directions, batch, hidden_dim)

        # For bidirectional: concatenate last forward + first backward
        if self.lstm.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]
        # hidden: (batch, hidden_dim * num_directions)

        output = self.fc(self.dropout(hidden))  # (batch, 1)
        return output.squeeze(1)


# Hyperparameters
VOCAB_SIZE  = len(vocab)
EMB_DIM     = 128
HIDDEN_DIM  = 256
OUTPUT_DIM  = 1
N_LAYERS    = 2
BIDIRECTIONAL = True
DROPOUT     = 0.5

model = LSTMClassifier(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM,
                        N_LAYERS, BIDIRECTIONAL, DROPOUT, PAD_IDX)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model architecture:\n{model}")
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 1.6 Training Loop

We use **Binary Cross-Entropy with Logits** (BCEWithLogitsLoss), which combines a sigmoid activation and binary cross-entropy in a single numerically stable operation:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[ y_i \log(\sigma(\hat{y}_i)) + (1 - y_i)\log(1 - \sigma(\hat{y}_i)) \right]$$

where $\hat{y}_i$ is the raw logit from our model and $\sigma$ is the sigmoid function.

**Optimizer:** Adam with learning rate $10^{-3}$.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    for texts, labels, lengths in tqdm(loader, desc='Training', leave=False):
        texts  = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(texts, lengths)
        loss = criterion(logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        epoch_loss += loss.item() * texts.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return epoch_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for texts, labels, lengths in tqdm(loader, desc='Evaluating', leave=False):
            texts  = texts.to(device)
            labels = labels.to(device)

            logits = model(texts, lengths)
            loss = criterion(logits, labels)

            epoch_loss += loss.item() * texts.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return epoch_loss / total, correct / total

In [ ]:
N_EPOCHS = 5
best_test_acc = 0
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    marker = ''
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_lstm_classifier.pt')
        marker = ' ✓ (saved)'

    print(f"Epoch {epoch}/{N_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  |  "
          f"Test Loss: {test_loss:.4f}  Acc: {test_acc:.4f}{marker}")

In [ ]:
# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train', marker='o')
ax1.plot(history['test_loss'],  label='Test',  marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', marker='o')
ax2.plot(history['test_acc'],  label='Test',  marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy over Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest test accuracy: {best_test_acc:.4f}")

## 1.7 Test on Custom Sentences

Let's see how our trained model performs on custom sentences we write ourselves.

In [ ]:
def predict_sentiment(model, text):
    """Predict sentiment of a single text string."""
    model.eval()
    indices = numericalize(text)
    tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)  # (1, seq_len)
    length = torch.tensor([len(indices)])

    with torch.no_grad():
        logit = model(tensor, length)
        prob = torch.sigmoid(logit).item()

    sentiment = 'Positive 👍' if prob >= 0.5 else 'Negative 👎'
    return sentiment, prob


test_sentences = [
    "This movie was absolutely wonderful! Great acting and a compelling story.",
    "Terrible film. The plot made no sense and the acting was wooden.",
    "An average movie, nothing special but not terrible either.",
    "I fell asleep halfway through. Boring and predictable.",
    "One of the best films I've seen this year. Highly recommend!",
    "The cinematography was beautiful but the script was weak.",
]

print("Sentiment Predictions\n" + "=" * 70)
for sent in test_sentences:
    sentiment, prob = predict_sentiment(model, sent)
    print(f"\n{sentiment}  (P(pos)={prob:.3f})")
    print(f"  → \"{sent[:80]}...\"" if len(sent) > 80 else f"  → \"{sent}\"")

---

# Part 2: Sequence-to-Sequence (Seq2Seq) Model

## 2.1 Theoretical Recap

### Encoder-Decoder Architecture

The Seq2Seq model (Sutskever et al., 2014) consists of two RNNs:
- **Encoder:** Reads the input sequence and compresses it into a fixed-size **context vector** $c$ (the final hidden state)
- **Decoder:** Takes the context vector and generates the output sequence one token at a time

```
Source: [x₁, x₂, ..., xₙ] → Encoder → context c → Decoder → [y₁, y₂, ..., yₘ]
```

### Bahdanau Attention (Preview)

The basic Seq2Seq model struggles with long sequences because the entire input must be compressed into a single vector $c$. **Attention** (Bahdanau et al., 2015) solves this by allowing the decoder to look at **all** encoder hidden states:

$$\alpha_{t,i} = \text{softmax}(\text{score}(s_t, h_i))$$

$$c_t = \sum_i \alpha_{t,i} \, h_i$$

where $s_t$ is the decoder state and $h_i$ are encoder hidden states. We will implement a basic Seq2Seq **without** attention first to understand the limitations, then observe them in Part 4.

### Our Task: Number Translation (English → French)

To keep training fast and the focus on architecture, we create a **synthetic** dataset of number translations:

| English | French |
|---------|--------|
| one two three | un deux trois |
| forty five | quarante cinq |
| one hundred | cent |

## 2.2 Build the Translation Dataset

We generate pairs of English → French number words (0–999). This gives us a controlled dataset where we can clearly evaluate whether the model learns the mapping.

In [ ]:
# --- Number-to-words dictionaries ---
EN_ONES = ['zero','one','two','three','four','five','six','seven','eight','nine',
           'ten','eleven','twelve','thirteen','fourteen','fifteen','sixteen',
           'seventeen','eighteen','nineteen']
EN_TENS = ['','','twenty','thirty','forty','fifty','sixty','seventy','eighty','ninety']

FR_ONES = ['zéro','un','deux','trois','quatre','cinq','six','sept','huit','neuf',
           'dix','onze','douze','treize','quatorze','quinze','seize',
           'dix-sept','dix-huit','dix-neuf']
FR_TENS = ['','','vingt','trente','quarante','cinquante','soixante',
           'soixante-dix','quatre-vingts','quatre-vingt-dix']

def num_to_en(n):
    if n < 20: return EN_ONES[n]
    if n < 100:
        return EN_TENS[n // 10] + ('' if n % 10 == 0 else ' ' + EN_ONES[n % 10])
    if n < 1000:
        rest = num_to_en(n % 100) if n % 100 != 0 else ''
        return EN_ONES[n // 100] + ' hundred' + (' ' + rest if rest else '')
    return str(n)

def num_to_fr(n):
    if n < 20: return FR_ONES[n]
    if n < 70:
        return FR_TENS[n // 10] + ('' if n % 10 == 0 else ' ' + FR_ONES[n % 10])
    if n < 80:  # 70-79: soixante-dix, soixante et onze, ...
        return 'soixante ' + FR_ONES[n - 60]
    if n < 100:  # 80-99: quatre-vingts, quatre-vingt-un, ...
        if n == 80: return 'quatre-vingts'
        return 'quatre-vingt ' + FR_ONES[n - 80]
    if n < 1000:
        prefix = '' if n // 100 == 1 else FR_ONES[n // 100] + ' '
        rest = num_to_fr(n % 100) if n % 100 != 0 else ''
        return prefix + 'cent' + (' ' + rest if rest else '')
    return str(n)

# Generate dataset
pairs = [(num_to_en(i).split(), num_to_fr(i).split()) for i in range(1000)]
random.shuffle(pairs)

# Show examples
print("Sample pairs (English → French):")
for en, fr in pairs[:8]:
    print(f"  {' '.join(en):30s} → {' '.join(fr)}")

print(f"\nTotal pairs: {len(pairs)}")

In [ ]:
# --- Build source and target vocabularies ---
SOS_TOKEN = '<sos>'  # Start of sequence
EOS_TOKEN = '<eos>'  # End of sequence

def build_seq2seq_vocab(pairs, side):
    """Build vocabulary from one side of the pairs (0=source, 1=target)."""
    word2idx = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
    for pair in pairs:
        for word in pair[side]:
            if word not in word2idx:
                word2idx[word] = len(word2idx)
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

src_w2i, src_i2w = build_seq2seq_vocab(pairs, 0)
tgt_w2i, tgt_i2w = build_seq2seq_vocab(pairs, 1)

SOS_IDX = 1
EOS_IDX = 2

print(f"Source vocab size: {len(src_w2i)}")
print(f"Target vocab size: {len(tgt_w2i)}")
print(f"Source words: {list(src_w2i.keys())[:15]}")
print(f"Target words: {list(tgt_w2i.keys())[:15]}")

In [ ]:
# --- Seq2Seq Dataset ---
class Seq2SeqDataset(Dataset):
    def __init__(self, pairs, src_w2i, tgt_w2i):
        self.data = []
        for src_words, tgt_words in pairs:
            src_ids = [src_w2i.get(w, 3) for w in src_words]
            tgt_ids = [SOS_IDX] + [tgt_w2i.get(w, 3) for w in tgt_words] + [EOS_IDX]
            self.data.append((torch.tensor(src_ids), torch.tensor(tgt_ids)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def seq2seq_collate(batch):
    srcs, tgts = zip(*batch)
    src_lens = torch.tensor([len(s) for s in srcs])
    tgt_lens = torch.tensor([len(t) for t in tgts])
    srcs_padded = pad_sequence(srcs, batch_first=True, padding_value=0)
    tgts_padded = pad_sequence(tgts, batch_first=True, padding_value=0)
    return srcs_padded, tgts_padded, src_lens, tgt_lens


# Train/test split
split = int(0.85 * len(pairs))
train_pairs = pairs[:split]
test_pairs  = pairs[split:]

train_s2s = Seq2SeqDataset(train_pairs, src_w2i, tgt_w2i)
test_s2s  = Seq2SeqDataset(test_pairs,  src_w2i, tgt_w2i)

train_s2s_loader = DataLoader(train_s2s, batch_size=64, shuffle=True,  collate_fn=seq2seq_collate)
test_s2s_loader  = DataLoader(test_s2s,  batch_size=64, shuffle=False, collate_fn=seq2seq_collate)

print(f"Train: {len(train_s2s)}, Test: {len(test_s2s)}")

## 2.3 Encoder, Decoder, and Seq2Seq Model

We implement the classic Seq2Seq architecture with GRU cells (lighter than LSTM, works well for this small task).

**Recall the GRU equations** from the slides:

| Gate | Formula | Role |
|------|---------|------|
| **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t])$ | How much of the old state to keep |
| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t])$ | How much of the old state to forget for candidate |
| **Candidate** | $\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])$ | New candidate hidden state |
| **Output** | $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$ | Interpolation between old and new |

**Training uses teacher forcing:** At each decoder step, we feed the **ground-truth** previous token (not the model's own prediction). This stabilizes training but can cause **exposure bias** (the model never learns to recover from its own mistakes).

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lens):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, emb_dim)
        packed = pack_padded_sequence(embedded, src_lens.cpu(),
                                     batch_first=True, enforce_sorted=False)
        outputs, hidden = self.rnn(packed)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        # hidden: (n_layers, batch, hidden_dim) — context vector
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden):
        # input_token: (batch, 1)
        embedded = self.dropout(self.embedding(input_token))  # (batch, 1, emb_dim)
        output, hidden = self.rnn(embedded, hidden)
        # output: (batch, 1, hidden_dim)
        prediction = self.fc_out(output.squeeze(1))  # (batch, vocab_size)
        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lens, tgt, teacher_forcing_ratio=0.5):
        # src: (batch, src_len), tgt: (batch, tgt_len)
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.fc_out.out_features

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # Encode
        _, hidden = self.encoder(src, src_lens)

        # First decoder input is <sos>
        input_token = tgt[:, 0:1]  # (batch, 1)

        for t in range(1, tgt_len):
            prediction, hidden = self.decoder(input_token, hidden)
            outputs[:, t] = prediction

            # Teacher forcing: use ground truth or model prediction
            if random.random() < teacher_forcing_ratio:
                input_token = tgt[:, t:t+1]  # Ground truth
            else:
                input_token = prediction.argmax(dim=1, keepdim=True)  # Model prediction

        return outputs


# Build model
S2S_EMB_DIM    = 64
S2S_HIDDEN_DIM = 128
S2S_N_LAYERS   = 1
S2S_DROPOUT    = 0.1

encoder = Encoder(len(src_w2i), S2S_EMB_DIM, S2S_HIDDEN_DIM, S2S_N_LAYERS, S2S_DROPOUT)
decoder = Decoder(len(tgt_w2i), S2S_EMB_DIM, S2S_HIDDEN_DIM, S2S_N_LAYERS, S2S_DROPOUT)
seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)

total_p = sum(p.numel() for p in seq2seq_model.parameters())
print(f"Seq2Seq model: {total_p:,} parameters")
print(seq2seq_model)

## 2.4 Training the Seq2Seq Model

We use **cross-entropy loss** between the decoder's predicted token distribution and the ground-truth target tokens. The `<pad>` token (index 0) is ignored in the loss computation.

In [ ]:
s2s_criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad>
s2s_optimizer = optim.Adam(seq2seq_model.parameters(), lr=1e-3)


def train_seq2seq_epoch(model, loader, criterion, optimizer, device, tf_ratio=0.5):
    model.train()
    epoch_loss = 0
    n_batches = 0
    for src, tgt, src_lens, tgt_lens in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()
        output = model(src, src_lens, tgt, teacher_forcing_ratio=tf_ratio)
        # output: (batch, tgt_len, vocab_size)
        # Reshape for CrossEntropy: skip first token (<sos>)
        output = output[:, 1:].reshape(-1, output.size(-1))
        tgt_flat = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
    return epoch_loss / n_batches


def eval_seq2seq(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    n_batches = 0
    with torch.no_grad():
        for src, tgt, src_lens, tgt_lens in loader:
            src = src.to(device)
            tgt = tgt.to(device)
            output = model(src, src_lens, tgt, teacher_forcing_ratio=0)  # No TF at eval
            output = output[:, 1:].reshape(-1, output.size(-1))
            tgt_flat = tgt[:, 1:].reshape(-1)
            loss = criterion(output, tgt_flat)
            epoch_loss += loss.item()
            n_batches += 1
    return epoch_loss / n_batches

In [ ]:
S2S_EPOCHS = 30
s2s_history = {'train_loss': [], 'test_loss': []}

for epoch in range(1, S2S_EPOCHS + 1):
    train_loss = train_seq2seq_epoch(seq2seq_model, train_s2s_loader,
                                     s2s_criterion, s2s_optimizer, device, tf_ratio=0.5)
    test_loss  = eval_seq2seq(seq2seq_model, test_s2s_loader, s2s_criterion, device)

    s2s_history['train_loss'].append(train_loss)
    s2s_history['test_loss'].append(test_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{S2S_EPOCHS}  Train Loss: {train_loss:.4f}  Test Loss: {test_loss:.4f}")

# Plot
plt.figure(figsize=(8, 4))
plt.plot(s2s_history['train_loss'], label='Train')
plt.plot(s2s_history['test_loss'],  label='Test')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Seq2Seq Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2.5 Greedy Decoding and Evaluation

At inference time, we use **greedy decoding**: at each step, the decoder picks the token with the highest probability and feeds it as input to the next step. Decoding stops when the model outputs `<eos>` or reaches a maximum length.

In [ ]:
def translate(model, sentence_words, src_w2i, tgt_i2w, device, max_len=20):
    """Translate a list of source words using greedy decoding."""
    model.eval()
    src_ids = [src_w2i.get(w, 3) for w in sentence_words]
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)  # (1, src_len)
    src_len = torch.tensor([len(src_ids)])

    with torch.no_grad():
        _, hidden = model.encoder(src_tensor, src_len)

        input_token = torch.tensor([[SOS_IDX]]).to(device)  # (1, 1)
        output_words = []

        for _ in range(max_len):
            prediction, hidden = model.decoder(input_token, hidden)
            top1 = prediction.argmax(dim=1)  # (1,)
            token_id = top1.item()

            if token_id == EOS_IDX:
                break
            output_words.append(tgt_i2w.get(token_id, '<unk>'))
            input_token = top1.unsqueeze(1)  # (1, 1)

    return output_words


# Test on training examples
print("Translation Results (Train Set Samples)")
print("=" * 60)
for en, fr in train_pairs[:10]:
    pred = translate(seq2seq_model, en, src_w2i, tgt_i2w, device)
    match = '✓' if pred == fr else '✗'
    print(f"  {match}  {' '.join(en):25s} → {' '.join(pred):25s} (expected: {' '.join(fr)})")

print(f"\nTranslation Results (Test Set Samples)")
print("=" * 60)
correct = 0
for en, fr in test_pairs:
    pred = translate(seq2seq_model, en, src_w2i, tgt_i2w, device)
    if pred == fr:
        correct += 1

print(f"Test accuracy (exact match): {correct}/{len(test_pairs)} = {correct/len(test_pairs):.1%}")

# Show some test examples
for en, fr in test_pairs[:10]:
    pred = translate(seq2seq_model, en, src_w2i, tgt_i2w, device)
    match = '✓' if pred == fr else '✗'
    print(f"  {match}  {' '.join(en):25s} → {' '.join(pred):25s} (expected: {' '.join(fr)})")

---

# Part 3: Visualizing Hidden States and Gradient Flow

## 3.1 Visualizing LSTM Hidden States

To understand **what** the LSTM learns, we can extract and visualize its hidden states at each time step. Each hidden state $h_t \in \mathbb{R}^{d}$ is a dense representation of the sequence up to position $t$.

We will:
1. Pass a review through our trained LSTM classifier
2. Extract $h_t$ at every time step
3. Create a heatmap showing which hidden dimensions activate for which words

In [ ]:
def extract_hidden_states(model, text, device):
    """Extract hidden states at every time step from our LSTM classifier."""
    model.eval()
    tokens = tokenize(text)[:50]  # Limit length for visualization
    indices = [vocab.get(t, UNK_IDX) for t in tokens]
    tensor = torch.tensor(indices).unsqueeze(0).to(device)  # (1, seq_len)
    length = torch.tensor([len(indices)])

    with torch.no_grad():
        embedded = model.embedding(tensor)
        packed = pack_padded_sequence(embedded, length.cpu(),
                                     batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = model.lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        # output: (1, seq_len, hidden_dim * num_directions)

    return tokens, output.squeeze(0).cpu().numpy()  # (seq_len, hidden_dim * 2)


# Choose a positive and a negative review
pos_text = "This film was absolutely brilliant. The acting was superb and the story was engaging from start to finish."
neg_text = "This movie was terrible. The plot was boring, the acting was awful, and I wasted two hours of my life."

pos_tokens, pos_hidden = extract_hidden_states(model, pos_text, device)
neg_tokens, neg_hidden = extract_hidden_states(model, neg_text, device)

# Plot heatmaps
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Select a subset of hidden dimensions for readability (first 50)
n_dims = 50

for ax, tokens, hidden, title in [
    (axes[0], pos_tokens, pos_hidden[:, :n_dims], 'Positive Review — LSTM Hidden States'),
    (axes[1], neg_tokens, neg_hidden[:, :n_dims], 'Negative Review — LSTM Hidden States')
]:
    im = ax.imshow(hidden.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Hidden Dimension')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.show()

print("\nObservation: Notice how certain hidden dimensions activate strongly on")
print("sentiment-bearing words (brilliant, superb, terrible, awful, boring).")
print("The LSTM learns to encode sentiment-relevant features in its hidden state.")

## 3.2 Visualizing Gradient Flow

One of the core issues with vanilla RNNs is the **vanishing gradient problem**: during backpropagation through time (BPTT), gradients are multiplied by the recurrent weight matrix at each step. If the spectral radius of $W_{hh}$ is less than 1, gradients shrink exponentially.

We will compare the gradient norms of a **vanilla RNN** vs. our **LSTM** to see this effect in practice. We:
1. Create a simple sequence task
2. Run a forward and backward pass
3. Record the gradient norm of the loss with respect to the hidden state at each time step

**Expected result:** For the vanilla RNN, gradient norms will decay rapidly for early time steps (information from the beginning of the sequence is lost). For the LSTM, gradients will be more stable thanks to the **additive cell state update**.

In [ ]:
def compute_gradient_flow(rnn_type='LSTM', seq_len=100, hidden_dim=64, input_dim=16):
    """
    Compute gradient norms of the loss w.r.t. intermediate hidden states
    for different RNN types.
    """
    torch.manual_seed(42)

    # Create a simple RNN
    if rnn_type == 'RNN':
        rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
    elif rnn_type == 'LSTM':
        rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
    elif rnn_type == 'GRU':
        rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)

    rnn.train()

    # Random input sequence
    x = torch.randn(1, seq_len, input_dim, requires_grad=True)

    # Forward pass — manually step through to record hidden states
    hidden_states = []

    if rnn_type == 'LSTM':
        h = torch.zeros(1, 1, hidden_dim)
        c = torch.zeros(1, 1, hidden_dim)
        for t in range(seq_len):
            inp = x[:, t:t+1, :]
            _, (h, c) = rnn(inp, (h, c))
            hidden_states.append(h)
    elif rnn_type == 'GRU':
        h = torch.zeros(1, 1, hidden_dim)
        for t in range(seq_len):
            inp = x[:, t:t+1, :]
            _, h = rnn(inp, h)
            hidden_states.append(h)
    else:  # vanilla RNN
        h = torch.zeros(1, 1, hidden_dim)
        for t in range(seq_len):
            inp = x[:, t:t+1, :]
            _, h = rnn(inp, h)
            hidden_states.append(h)

    # Loss based on the final hidden state
    loss = hidden_states[-1].sum()

    # Compute gradients
    loss.backward()

    # Collect gradient norms for each time step
    grad_norms = []
    for t in range(seq_len):
        if hidden_states[t].grad is not None:
            grad_norms.append(hidden_states[t].grad.norm().item())
        else:
            # Compute gradient manually via autograd
            grad = torch.autograd.grad(loss, hidden_states[t], retain_graph=True)[0]
            grad_norms.append(grad.norm().item())

    return grad_norms


# Compare gradient flow across RNN types
SEQ_LEN = 100
rnn_grads  = compute_gradient_flow('RNN',  seq_len=SEQ_LEN)
lstm_grads = compute_gradient_flow('LSTM', seq_len=SEQ_LEN)
gru_grads  = compute_gradient_flow('GRU',  seq_len=SEQ_LEN)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
ax.plot(range(SEQ_LEN), rnn_grads,  label='Vanilla RNN', alpha=0.8)
ax.plot(range(SEQ_LEN), lstm_grads, label='LSTM', alpha=0.8)
ax.plot(range(SEQ_LEN), gru_grads,  label='GRU', alpha=0.8)
ax.set_xlabel('Time Step')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Flow — Linear Scale')
ax.legend()
ax.grid(True, alpha=0.3)

# Log scale
ax = axes[1]
ax.plot(range(SEQ_LEN), rnn_grads,  label='Vanilla RNN', alpha=0.8)
ax.plot(range(SEQ_LEN), lstm_grads, label='LSTM', alpha=0.8)
ax.plot(range(SEQ_LEN), gru_grads,  label='GRU', alpha=0.8)
ax.set_xlabel('Time Step')
ax.set_ylabel('Gradient Norm (log scale)')
ax.set_title('Gradient Flow — Log Scale')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print ratio
print(f"\nGradient ratio (step 0 / step {SEQ_LEN-1}):")
print(f"  Vanilla RNN: {rnn_grads[0] / rnn_grads[-1]:.6f}")
print(f"  LSTM:        {lstm_grads[0] / lstm_grads[-1]:.6f}")
print(f"  GRU:         {gru_grads[0] / gru_grads[-1]:.6f}")

### Interpretation

The plot above demonstrates the **vanishing gradient problem**:

- **Vanilla RNN:** Gradient norms decay **exponentially** as we go back in time. By time step 0, the gradient is essentially zero — the model cannot learn from the beginning of the sequence.
- **LSTM:** Gradient norms remain much more **stable** across time steps, thanks to the additive cell state update $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$. The gradient can flow through $C_t$ undiminished when $f_t \approx 1$.
- **GRU:** Similar behavior to LSTM, with slightly different gradient dynamics due to its simpler gate structure.

This is exactly why LSTMs and GRUs were introduced — they can effectively learn dependencies spanning **hundreds** of time steps, while vanilla RNNs fail beyond ~10-20 steps.

---

# Part 4: Limitations of RNNs on Long Sequences

## 4.1 The Copy Task — A Diagnostic Benchmark

To empirically show the limitations of RNNs on long sequences, we use the **copy task**: the model must remember a short pattern presented at the **beginning** of a sequence and reproduce it after a long delay filled with zeros.

```
Input:  [3, 1, 4, 1, 5, 0, 0, 0, ..., 0, 0, 0]   (pattern + T zeros)
Target: [0, 0, 0, 0, 0, 0, 0, 0, ..., 3, 1, 4, 1, 5]   (zeros + pattern)
```

As the delay $T$ increases, the model must remember the pattern over longer and longer periods. Vanilla RNNs fail quickly, while LSTMs can handle longer delays — but eventually all RNN variants struggle when $T$ becomes very large.

In [ ]:
def generate_copy_data(n_samples, pattern_len, delay, n_symbols=8):
    """
    Generate copy-task data.
    Input:  [pattern..., 0 * delay]
    Target: [0 * (pattern_len + delay - pattern_len), pattern...]
    Symbols are 1..n_symbols (0 is blank).
    """
    total_len = pattern_len + delay
    inputs = torch.zeros(n_samples, total_len, dtype=torch.long)
    targets = torch.zeros(n_samples, total_len, dtype=torch.long)

    patterns = torch.randint(1, n_symbols + 1, (n_samples, pattern_len))
    inputs[:, :pattern_len] = patterns
    targets[:, -pattern_len:] = patterns

    return inputs, targets


class CopyModel(nn.Module):
    def __init__(self, n_symbols, hidden_dim, rnn_type='LSTM'):
        super().__init__()
        self.embedding = nn.Embedding(n_symbols + 1, hidden_dim)  # +1 for blank (0)
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.RNN(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_symbols + 1)

    def forward(self, x):
        emb = self.embedding(x)
        output, _ = self.rnn(emb)
        logits = self.fc(output)
        return logits


def train_copy_task(rnn_type, delay, n_epochs=200, pattern_len=5,
                    hidden_dim=64, n_symbols=8, lr=1e-3):
    """Train a model on the copy task and return final accuracy."""
    model = CopyModel(n_symbols, hidden_dim, rnn_type).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = generate_copy_data(500, pattern_len, delay, n_symbols)
    test_x,  test_y  = generate_copy_data(200, pattern_len, delay, n_symbols)
    train_x, train_y = train_x.to(device), train_y.to(device)
    test_x,  test_y  = test_x.to(device),  test_y.to(device)

    losses = []
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(train_x)  # (batch, total_len, n_symbols+1)
        loss = criterion(logits.reshape(-1, n_symbols + 1), train_y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())

    # Evaluate: accuracy on the PATTERN portion only
    model.eval()
    with torch.no_grad():
        preds = model(test_x).argmax(dim=-1)
        # Only check the last pattern_len positions
        pattern_preds = preds[:, -pattern_len:]
        pattern_truth = test_y[:, -pattern_len:]
        acc = (pattern_preds == pattern_truth).float().mean().item()

    return acc, losses


print("Example copy task (pattern_len=5, delay=10):")
inp, tgt = generate_copy_data(1, 5, 10)
print(f"Input:  {inp[0].tolist()}")
print(f"Target: {tgt[0].tolist()}")

In [ ]:
# --- Compare RNN types across increasing delays ---
delays = [10, 25, 50, 100, 200]
rnn_types = ['RNN', 'GRU', 'LSTM']
results = {rnn_type: [] for rnn_type in rnn_types}

print("Training copy task across delays (this may take a few minutes)...")
print(f"{'Delay':>6s}", end='')
for rnn_type in rnn_types:
    print(f"  {rnn_type:>8s}", end='')
print()
print("-" * 40)

for delay in delays:
    print(f"{delay:6d}", end='', flush=True)
    for rnn_type in rnn_types:
        acc, _ = train_copy_task(rnn_type, delay, n_epochs=300)
        results[rnn_type].append(acc)
        print(f"  {acc:8.1%}", end='', flush=True)
    print()

In [ ]:
# --- Plot results ---
plt.figure(figsize=(10, 5))

markers = {'RNN': 'o', 'GRU': 's', 'LSTM': '^'}
colors  = {'RNN': '#e74c3c', 'GRU': '#2ecc71', 'LSTM': '#3498db'}

for rnn_type in rnn_types:
    plt.plot(delays, results[rnn_type], marker=markers[rnn_type],
             color=colors[rnn_type], label=rnn_type, linewidth=2, markersize=8)

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect')
plt.xlabel('Delay Length (number of blank steps)', fontsize=12)
plt.ylabel('Copy Accuracy (pattern portion)', fontsize=12)
plt.title('Copy Task — RNN Limitations on Long Sequences', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

### Interpretation

The copy task results clearly illustrate **the sequential memory bottleneck** of RNNs:

| Observation | Explanation |
|-------------|-------------|
| **Vanilla RNN fails first** | Vanishing gradients make it impossible to carry information across many blank steps |
| **GRU holds longer** | The update gate $z_t$ can learn to hold information when the input is blank ($z_t \approx 0$ keeps old state) |
| **LSTM holds longest** | The dedicated cell state $C_t$ provides an explicit memory highway, but it too degrades with very long delays |
| **All eventually fail** | No RNN variant can perfectly handle arbitrarily long dependencies — this is a fundamental limitation |

This is precisely the motivation for the **Transformer architecture** (Part 2.2 of the course), which uses **self-attention** to access any position in the sequence directly:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

With attention, the model can attend to position 0 and position 1000 equally well — there is no sequential bottleneck.

## 4.2 Inference Speed — Sequential Bottleneck

Beyond gradient flow, RNNs have another fundamental limitation: they process sequences **sequentially**. Each hidden state $h_t$ depends on $h_{t-1}$, making parallelization impossible along the time axis. Let's measure this.

In [ ]:
import time

def benchmark_rnn_speed(seq_lengths, hidden_dim=256, input_dim=128, n_runs=10):
    """Benchmark RNN inference time across sequence lengths."""
    rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True, num_layers=2).to(device)
    rnn.eval()

    times = []
    for seq_len in seq_lengths:
        x = torch.randn(1, seq_len, input_dim).to(device)

        # Warm up
        with torch.no_grad():
            _ = rnn(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        # Timed runs
        t0 = time.perf_counter()
        for _ in range(n_runs):
            with torch.no_grad():
                _ = rnn(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
        elapsed = (time.perf_counter() - t0) / n_runs
        times.append(elapsed * 1000)  # ms

    return times


seq_lengths = [32, 64, 128, 256, 512, 1024, 2048]
rnn_times = benchmark_rnn_speed(seq_lengths)

plt.figure(figsize=(8, 5))
plt.plot(seq_lengths, rnn_times, 'o-', color='#3498db', linewidth=2, markersize=8)
plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Inference Time (ms)', fontsize=12)
plt.title('LSTM Inference Time vs. Sequence Length', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Sequence Length → Inference Time:")
for sl, t in zip(seq_lengths, rnn_times):
    print(f"  {sl:5d} tokens → {t:8.2f} ms")

print(f"\nTime grows approximately linearly — O(T) — because each step")
print(f"depends on the previous one. Transformers process all positions")
print(f"in parallel, making them O(1) in depth (though O(T²) in memory).")

---

# Summary

## What We Built and Learned

| Part | Task | Key Takeaway |
|------|------|--------------|
| **Part 1** | LSTM Sentiment Classifier on IMDB | Bidirectional LSTM + packed sequences can achieve ~85%+ accuracy. Embedding → LSTM → Linear is the standard recipe. |
| **Part 2** | Seq2Seq Number Translation (EN→FR) | Encoder compresses input to a context vector; decoder generates output autoregressively. Teacher forcing helps training but causes exposure bias. |
| **Part 3** | Hidden State & Gradient Visualization | LSTM hidden states encode sentiment features. Vanilla RNN gradients vanish exponentially; LSTM/GRU gradients remain stable. |
| **Part 4** | Copy Task & Speed Benchmark | All RNN variants eventually fail on very long-range dependencies. Inference time scales linearly with sequence length — sequential bottleneck. |

## Why This Matters for the Course

The limitations we observed in this lab — **vanishing gradients**, **information bottleneck** in Seq2Seq, and **sequential processing** — are precisely the problems that the **Transformer architecture** (Section 2.2) was designed to solve:

- **Self-attention** replaces recurrence → parallel processing, $O(1)$ path length between any two positions
- **No sequential bottleneck** → the full input is available to every layer
- **Multi-head attention** → the model can attend to different aspects of the input simultaneously

In **Lab 3**, we will implement Transformer-based models and directly compare their performance with the RNN models we built here.

---

## Exercises (To Go Further)

1. **Add Bahdanau attention** to the Seq2Seq model and compare translation accuracy. Use the formulas: $\alpha_{t,i} = \text{softmax}(\text{score}(s_t, h_i))$ and $c_t = \sum_i \alpha_{t,i} h_i$.
2. **Try GloVe embeddings** in the LSTM classifier — replace `nn.Embedding` with pre-trained vectors and compare accuracy.
3. **Experiment with GRU** instead of LSTM in the classifier — is performance similar? How does training speed compare?
4. **Increase the copy task delay** to 500+ and observe at what point even LSTM fails completely.
5. **Beam search**: Implement beam search decoding for the Seq2Seq model instead of greedy decoding.